# Pengukuran indeks OpenSearch: 

Tujuan: bandingkan tiga jenis ukuran untuk dokumen yang sama.

1. **Ukuran dokumen mentah**: ukuran dokumen sebelum ditambahkan embedding.
2. **Ukuran estimasi index**: ukuran dokumen setelah ditambahkan embedding jumlah vektor × dimensi × 4 byte float32.
3. **Ukuran index aktual**: ukuran penyimpanan primary yang dilaporkan
   oleh OpenSearch. Mencakup dokumen, vektor, graph HNSW, dan struktur Lucene.

In [33]:
from __future__ import annotations

import json
from collections.abc import Iterable
from pathlib import Path
from typing import Any

import pandas as pd
from fastembed import TextEmbedding
from IPython.display import display
from opensearchpy import OpenSearch, helpers

from index_memory_usage import (
    calculate_raw_documents_size,
    estimate_raw_embedding_bytes,
    get_opensearch_index_size_bytes,
)

## 1. Konfigurasi

In [34]:
def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current directory")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = (
    PROJECT_ROOT / "scripts" / "scraping" / "data" / "brin" / "datasets.ndjson"
)

OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_URL = f"http://{OPENSEARCH_HOST}:{OPENSEARCH_PORT}"

# This is a dedicated disposable index so each run measures a clean index.
INDEX_NAME = "brin-datasets-size-demo"
SAMPLE_SIZE = 1_000
EMBEDDING_BATCH_SIZE = 32
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BYTES_PER_DIMENSION = 4

QUERY_TEXT = "karakteristik morfologi tanaman Piper"
RESULT_COUNT = 5
METADATA_FILTERS: dict[str, str] = {
    "author": "Astuti, Inggit Puji",
    "institution": "Pusat Konservasi Tumbuhan Kebun Raya",
    "topic": "Morfologi",
}

## 2. Load records dari dataset metadata BRIN Dataverse

In [35]:
records: list[dict[str, Any]] = []

with DATA_PATH.open(encoding="utf-8") as source:
    for line_number, line in enumerate(source, start=1):
        if len(records) >= SAMPLE_SIZE:
            break
        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid JSON on line {line_number} of {DATA_PATH}"
            ) from error

        if isinstance(record, dict) and record.get("description"):
            records.append(record)

if not records:
    raise RuntimeError(f"No records with descriptions found in {DATA_PATH}")

print(f"Loaded {len(records):,} records from {DATA_PATH.name}.")

Loaded 1,000 records from datasets.ndjson.


In [36]:
record_preview = pd.DataFrame(
    {
        "name": record.get("name"),
        "description_preview": f"{record['description'][:160]}...",
        "authors": record.get("authors", []),
        "topics": [
            *(record.get("keywords") or []),
            *(record.get("subjects") or []),
        ],
    }
    for record in records[:3]
)
display(record_preview)

,name,description_preview,authors,topics
0,Karakteristik Morfologi Piper Acutilimbum C. D...,Piper acutilimbum C. DC dan Piper baccatum Blu...,"[Astuti, Inggit Puji, Munawaroh, Esti ]","[Morfologi, Medicine, Health and Life Sciences]"
1,Studi Pendahuluan Pelindian Kolom Bijih Mangan...,Bijih mangan merupakan bahan tambang yang pent...,"[Sumardi, Slamet , M, Fika Rofiek, Prasetyo, E...","[Pelindian, Asam oksalat, Column leaching, Bi..."
2,Kultur Tunas Pucuk Anggrek Endemik Terancam Pu...,"The history of Cymbidium hartinahianum, a terr...","[Mursidawati, Sofi, Handini, Elizabeth, Rahay...","[Medicine, Health and Life Sciences]"


## 3. Membuat embedding untuk setiap deskripsi 

Setiap record menghasilkan satu embedding. Teks tidak dipecah menjadi chunk.

In [37]:
model = TextEmbedding(model_name=EMBEDDING_MODEL)
embeddings = [
    embedding.tolist()
    for embedding in model.embed(
        (record["description"] for record in records),
        batch_size=EMBEDDING_BATCH_SIZE,
    )
]

vector_dimension = len(embeddings[0])
print(f"Created {len(embeddings):,} vectors with {vector_dimension:,} dimensions.")

Created 1,000 vectors with 384 dimensions.


## 4. Transformasi dokumen yang akan dikirim ke OpenSearch

In [38]:
def unique_strings(values: Iterable[object]) -> list[str]:
    result: list[str] = []
    seen: set[str] = set()
    for value in values:
        if not isinstance(value, str):
            continue
        normalized = value.strip()
        if normalized and normalized not in seen:
            seen.add(normalized)
            result.append(normalized)
    return result


def citation_fields(record: dict[str, Any]) -> list[dict[str, Any]]:
    fields = record.get("metadataBlocks", {}).get("citation", {}).get("fields", [])
    return fields if isinstance(fields, list) else []


def author_metadata(record: dict[str, Any], field_name: str) -> list[str]:
    values: list[object] = []
    for field in citation_fields(record):
        if field.get("typeName") != "author":
            continue
        for author in field.get("value", []):
            nested_field = author.get(field_name, {})
            values.append(nested_field.get("value"))
    return unique_strings(values)


def prepare_document(record: dict[str, Any], embedding: list[float]) -> dict[str, Any]:
    authors = unique_strings(record.get("authors") or [])
    if not authors:
        authors = author_metadata(record, "authorName")

    contact_affiliations = [
        contact.get("affiliation")
        for contact in (record.get("contacts") or [])
        if isinstance(contact, dict)
    ]
    institutions = unique_strings(
        [*author_metadata(record, "authorAffiliation"), *contact_affiliations]
    )
    topics = unique_strings(
        [*(record.get("keywords") or []), *(record.get("subjects") or [])]
    )

    return {
        "name": record.get("name"),
        "description": record["description"],
        "description_embedding": embedding,
        "embedding_model": EMBEDDING_MODEL,
        "author": authors,
        "institution": institutions,
        "topic": topics,
        "url": record.get("url"),
        "global_id": record.get("global_id"),
    }


documents = [
    prepare_document(record, embedding)
    for record, embedding in zip(records, embeddings)
]

## 5. Menghitung dua ukuran sebelum proses indexing

`calculate_raw_documents_size` tidak menghitung field `description_embedding`,
sehingga hasilnya hanya mencakup field JSON non-vektor yang dikirim ke
OpenSearch.

`estimate_raw_embedding_bytes` sebaliknya hanya menghitung `description_embedding`dan tidak mencakup field JSON non-vektor.

Hasi keduanya perlu ditambahkan untuk mendapatkan ukuran estimasi index.

In [39]:
raw_document_stats = calculate_raw_documents_size(documents)
estimated_raw_embedding_bytes = estimate_raw_embedding_bytes(
    num_vectors=len(documents),
    dimensions=vector_dimension,
    bytes_per_dimension=BYTES_PER_DIMENSION,
)

estimated_raw_documents_with_embeddings_bytes = (
    raw_document_stats["total_size_in_bytes"] + estimated_raw_embedding_bytes
)

print(f"Raw document JSON: {raw_document_stats['total_size_in_bytes']:,} bytes")
print(f"Estimated raw embeddings: {estimated_raw_embedding_bytes:,} bytes")
print(
    "Estimated raw documents with embeddings: "
    f"{estimated_raw_documents_with_embeddings_bytes:,} bytes"
)

Raw document JSON: 1,960,979 bytes
Estimated raw embeddings: 1,536,000 bytes
Estimated raw documents with embeddings: 3,496,979 bytes


## 6. Membuat indeks OpenSearch

Indeks dihapus lalu dibuat ulang setiap kali notebook dijalankan dari
awal. 

In [40]:
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    use_ssl=False,
    verify_certs=False,
)

cluster_info = client.info()
print(
    f"Connected to OpenSearch {cluster_info['version']['number']} "
    f"at {OPENSEARCH_URL}."
)

Connected to OpenSearch 3.8.0 at http://localhost:9200.


In [41]:
def index_mapping(dimension: int) -> dict[str, Any]:
    return {
        "settings": {
            "index": {
                "knn": True,
                "number_of_shards": 1,
                "number_of_replicas": 0,
            }
        },
        "mappings": {
            "properties": {
                "name": {"type": "text"},
                "description": {"type": "text"},
                "description_embedding": {
                    "type": "knn_vector",
                    "dimension": dimension,
                    "method": {
                        "name": "hnsw",
                        "space_type": "cosinesimil",
                        "engine": "lucene",
                    },
                },
                "embedding_model": {"type": "keyword"},
                "author": {"type": "keyword"},
                "institution": {"type": "keyword"},
                "topic": {"type": "keyword"},
                "url": {"type": "keyword", "index": False},
                "global_id": {"type": "keyword"},
            }
        },
    }


if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)

client.indices.create(
    index=INDEX_NAME,
    body=index_mapping(vector_dimension),
)

{'acknowledged': True,
 'shards_acknowledged': True,
 'index': 'brin-datasets-size-demo'}

## 7. Indexing dokumen ke OpenSearch

In [42]:
actions = [
    {
        "_op_type": "index",
        "_index": INDEX_NAME,
        "_id": record.get("global_id") or str(position),
        "_source": document,
    }
    for position, (record, document) in enumerate(zip(records, documents))
]

indexed_count, _ = helpers.bulk(client, actions, refresh=True)
print(f"Indexed {indexed_count:,} documents into {INDEX_NAME!r}.")

Indexed 1,000 documents into 'brin-datasets-size-demo'.


In [43]:
actual_index_size_bytes = get_opensearch_index_size_bytes(
    base_url=OPENSEARCH_URL,
    index_name=INDEX_NAME,
)

print(f"Actual OpenSearch primary index: {actual_index_size_bytes:,} bytes")

Actual OpenSearch primary index: 3,753,624 bytes


## 8. Sample query

In [44]:
def metadata_filter(filters: dict[str, str]) -> dict[str, Any] | None:
    clauses = [
        {"term": {field: value}}
        for field, value in filters.items()
        if value
    ]
    if not clauses:
        return None
    return {"bool": {"filter": clauses}}


def search(
    client: OpenSearch,
    model: TextEmbedding,
    query_text: str,
    filters: dict[str, str],
) -> list[dict[str, Any]]:
    query_vector = next(model.query_embed(query_text)).tolist()
    knn_query: dict[str, Any] = {
        "vector": query_vector,
        "k": RESULT_COUNT,
    }
    if filter_query := metadata_filter(filters):
        knn_query["filter"] = filter_query

    response = client.search(
        index=INDEX_NAME,
        body={
            "size": RESULT_COUNT,
            "_source": {"excludes": ["description_embedding"]},
            "query": {"knn": {"description_embedding": knn_query}},
        },
    )
    return response["hits"]["hits"]


hits = search(client, model, QUERY_TEXT, METADATA_FILTERS)
search_results = pd.DataFrame(
    {
        "score": hit["_score"],
        "name": hit["_source"]["name"],
        "authors": ", ".join(hit["_source"]["author"]),
        "institutions": ", ".join(hit["_source"]["institution"]),
        "topics": ", ".join(hit["_source"]["topic"]),
        "url": hit["_source"]["url"],
    }
    for hit in hits
)
display(search_results)

,score,name,authors,institutions,topics,url
0,0.923398,Karakteristik Morfologi Piper Acutilimbum C. D...,"Astuti, Inggit Puji, Munawaroh, Esti",Pusat Konservasi Tumbuhan Kebun Raya,"Morfologi, Medicine, Health and Life Sciences",https://hdl.handle.net/20.500.12690/RIN/02VX0O


## 9. Perbandingan akhir

In [48]:
model_info = pd.DataFrame(
    [
        {
            "embedding_model": EMBEDDING_MODEL,
            "document_count": indexed_count,
            "vector_dimensions": vector_dimension,
            "vector_data_type": "float32",
            "bytes_per_dimension": BYTES_PER_DIMENSION,
            # "chunking": "none",
        }
    ]
)

size_comparison = pd.DataFrame(
    [
        {
            "measurement": "Ukuran dokumen mentah",
            "description": "Ukuran dokumen mentah dalam format JSON sebelum ditambahkan embedding.",
            "size_bytes": raw_document_stats["total_size_in_bytes"],
        },
        {
            "measurement": "Ukuran estimasi embedding",
            "description": "Ukuran estimasi embedding berdasarkan jumlah dokumen, dimensi vektor, dan bytes per dimensi.",
            "size_bytes": estimated_raw_embedding_bytes,
        },
                {
            "measurement": "Ukuran estimasi index",
            "description": "Ukuran estimasi index OpenSearch berdasarkan gabungan ukuran dokumen mentah dan estimasi embedding.",
            "size_bytes": estimated_raw_documents_with_embeddings_bytes,
        },
        {
            "measurement": "Ukuran index OpenSearch aktual",
            "description": "Ukuran index OpenSearch aktual setelah dokumen diindeks.",
            "size_bytes": actual_index_size_bytes,
        },
    ]
)

size_comparison["size_mib"] = size_comparison["size_bytes"] / (1024**2)
size_comparison["size_gib"] = size_comparison["size_bytes"] / (1024**3)
# size_comparison["bytes_per_document"] = (
#     size_comparison["size_bytes"] / indexed_count
# )
# size_comparison["percent_of_actual_index"] = (
#     size_comparison["size_bytes"] / actual_index_size_bytes * 100
# )

display(model_info)
display(
    size_comparison.style.format(
        {
            "size_bytes": "{:,.0f}",
            "size_mib": "{:,.3f}",
            "size_gib": "{:,.6f}",
            "bytes_per_document": "{:,.1f}",
            # "percent_of_actual_index": "{:,.1f}%",
        }
    )
)

,embedding_model,document_count,vector_dimensions,vector_data_type,bytes_per_dimension
0,sentence-transformers/paraphrase-multilingual-...,1000,384,float32,4


,measurement,description,size_bytes,size_mib,size_gib
0,Ukuran dokumen mentah,Ukuran dokumen mentah dalam format JSON sebelum ditambahkan embedding.,"1,960,979",1.870,0.001826
1,Ukuran estimasi embedding,"Ukuran estimasi embedding berdasarkan jumlah dokumen, dimensi vektor, dan bytes per dimensi.","1,536,000",1.465,0.001431
2,Ukuran estimasi index,Ukuran estimasi index OpenSearch berdasarkan gabungan ukuran dokumen mentah dan estimasi embedding.,"3,496,979",3.335,0.003257
3,Ukuran index OpenSearch aktual,Ukuran index OpenSearch aktual setelah dokumen diindeks.,"3,753,624",3.580,0.003496
